# Stream Alpha – M20 Training on Colab

Train NHITS + PatchTST ensemble on BTC/USD, ETH/USD, SOL/USD using NeuralForecast.

**Before running:**
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Upload your exported parquet dataset to Google Drive at:
   `My Drive/Stream_Alpha/feature_ohlc_for_colab/`


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATASET_DIR = '/content/drive/MyDrive/Stream_Alpha/feature_ohlc_for_colab'
assert os.path.isdir(DATASET_DIR), f'Dataset not found at {DATASET_DIR}'
print('Dataset folders:', os.listdir(DATASET_DIR))


## 2. Clone the repo

In [ ]:
!git clone https://github.com/Arashmehrdad/Stream_Alpha.git /content/Stream_Alpha
%cd /content/Stream_Alpha


## 3. Install dependencies

In [ ]:
!pip install -q \
    'neuralforecast>=1.7,<2' \
    'lightning>=2.2,<3' \
    'autogluon.tabular[all]==1.5.0' \
    'scikit-learn>=1.6,<1.7' \
    'numpy>=1.26,<2' \
    'pyarrow' \
    'asyncpg>=0.29,<0.30'

print('\nDependencies installed!')


## 4. Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

import psutil
print(f'System RAM: {psutil.virtual_memory().total / 1e9:.1f} GB')


## 5. Set artifact output to Google Drive

Artifacts are saved to Drive so they survive runtime disconnections.

In [ ]:
import json
from pathlib import Path

config_path = Path('/content/Stream_Alpha/configs/training.m20.json')
config = json.loads(config_path.read_text())

DRIVE_ARTIFACTS = '/content/drive/MyDrive/Stream_Alpha/artifacts/training/m20'
config['artifact_root'] = DRIVE_ARTIFACTS
os.makedirs(DRIVE_ARTIFACTS, exist_ok=True)

COLAB_CONFIG = '/content/Stream_Alpha/configs/training.m20.colab.json'
Path(COLAB_CONFIG).write_text(json.dumps(config, indent=2))
print(f'Artifacts will be saved to {DRIVE_ARTIFACTS}')


## 6. Run M20 Training

Loads data from parquet files (no PostgreSQL needed) and runs the full
walk-forward evaluation with NHITS + PatchTST.

In [ ]:
!cd /content/Stream_Alpha && python -m app.training \
    --config configs/training.m20.colab.json \
    --parquet-dir "$DATASET_DIR"


## 7. Check artifacts

In [ ]:
for root, dirs, files in os.walk(DRIVE_ARTIFACTS):
    level = root.replace(DRIVE_ARTIFACTS, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:
        print(f'{subindent}{file}')
    if len(files) > 10:
        print(f'{subindent}... and {len(files) - 10} more')


## 8. Resume from checkpoint (optional)

If the runtime disconnects, reconnect and run cells 1-5, then this cell instead of cell 6:

In [ ]:
# Uncomment to resume from a partial run:
# !cd /content/Stream_Alpha && python -m app.training \
#     --config configs/training.m20.colab.json \
#     --parquet-dir "$DATASET_DIR" \
#     --resume "$DRIVE_ARTIFACTS"
